# Étape 6 — Diagnostic direct de l'index FAISS

**Objectif :** vérifier, sans passer par le LLM, si une information précise (ex:
"Kofax Express") existe réellement dans l'index, et à quel rang elle ressort pour
une requête donnée. Permet de distinguer un problème d'indexation (le contenu
n'est pas là) d'un problème de classement (le contenu est là mais mal classé) ou
de génération (le LLM ignore une bonne source malgré tout).

**Kernel à utiliser :** `Python (RAG GED)`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from build_index import EMBEDDING_MODEL_NAME, INDEX_DIR

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
print("Index chargé.")

C:\Users\DELL\AppData\Local\Temp\ipykernel_3268\1670282059.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index chargé.


## 1. Le texte recherché existe-t-il vraiment dans un chunk indexé ?

Recherche textuelle brute (pas de similarité, juste "est-ce que le mot est présent")
— la vérité de base, indépendante de tout modèle.

In [2]:
MOT_CLE = "kofax"  # change ici pour tester un autre mot (ex: "readsoft")

docstore = vectorstore.docstore._dict
chunks_trouves = [
    (doc_id, doc) for doc_id, doc in docstore.items()
    if MOT_CLE.lower() in doc.page_content.lower()
]

print(f"Nombre de chunks contenant '{MOT_CLE}' : {len(chunks_trouves)}\n")
for doc_id, doc in chunks_trouves:
    print(f"--- Source : {doc.metadata.get('source')} ---")
    print(doc.page_content)
    print()

Nombre de chunks contenant 'kofax' : 4

--- Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf ---
2.4.6 Archivage . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 12
2.5 Processus spécifique CMIM . . . . . . . . . . . . . . . . . . . . . . . . . . 12
2.5.1 Particularités . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 13
2.5.2 Types de dossiers . . . . . . . . . . . . . . . . . . . . . . . . . . . . 13
2.5.3 Utilisation de Kofax Express . . . . . . . . . . . . . . . . . . . . . . 13
2.6 Processus documentaire multi-clients . . . . . . . . . . . . . . . . . . . . . 14
2.6.1 RMA . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 14
2.6.2 BANK OF AFRICA . . . . . . . . . . . . . . . . . . . . . . . . . . 14
2.6.3 SALAFIN . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 14

--- Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf ---
dossier co

## 2. À quel rang ce chunk ressort-il pour une vraie question ?

Si le chunk existe (étape 1) mais n'apparaît pas dans le top 5-10 ici, c'est un
problème de classement (retrieval), pas d'indexation — il faut alors soit
augmenter `k`, soit reformuler différemment le contenu source pour qu'il "matche"
mieux sémantiquement la question.

In [3]:
QUESTION_TEST = "quel logiciel est utilisé pour numériser les dossiers CMIM"

resultats = vectorstore.similarity_search_with_score(QUESTION_TEST, k=10)

print(f"Question : {QUESTION_TEST}\n")
for rang, (doc, score) in enumerate(resultats, start=1):
    marqueur = "✅ CONTIENT KOFAX" if "kofax" in doc.page_content.lower() else ""
    print(f"{rang}. [score={score:.3f}] {doc.metadata.get('source')} {marqueur}")
    print(f"   {doc.page_content[:150]}...")
    print()

Question : quel logiciel est utilisé pour numériser les dossiers CMIM

1. [score=5.744] C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf 
   Les informations relatives aux dates de réception, de traitement et d’archivage sont
également enregistrées dans le système.
Cette organisation facili...

2. [score=5.758] C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf 
   SALAFIN et autres partenaires) repose quant à lui sur une application distincte de celle
utilisée pour les dossiers de la CMIM, mais dont l’interface ...

3. [score=6.790] C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf ✅ CONTIENT KOFAX
   2.7.1 Tableau comparatif
Critère Processus CMIM Processus multi-
clients
Clients concernés CMIM RMA, BANK OF
AFRICA, SALAFIN et
autres partenaires
Org...

4. [score=7.122] C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf ✅ CONTIENT KOFAX
   doss

## 3. Vérifier aussi ReadSoft Forms (pour la question RMA)

In [4]:
MOT_CLE_2 = "readsoft"

chunks_readsoft = [
    (doc_id, doc) for doc_id, doc in docstore.items()
    if MOT_CLE_2.lower() in doc.page_content.lower()
]
print(f"Nombre de chunks contenant '{MOT_CLE_2}' : {len(chunks_readsoft)}\n")
for doc_id, doc in chunks_readsoft:
    print(f"--- Source : {doc.metadata.get('source')} ---")
    print(doc.page_content)
    print()

Nombre de chunks contenant 'readsoft' : 3

--- Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf ---
2.6.3 SALAFIN . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 14
2.6.4 ReadSoft Forms . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
2.6.5 Interpret . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
2.6.6 Transfer . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
2.6.7 Vidéocodage . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
2.7 Analyse comparative . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
2.7.1 Tableau comparatif . . . . . . . . . . . . . . . . . . . . . . . . . . . 16
2.7.2 Avantages . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 16
2.7.3 Différences . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 17

--- Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf --

## Interprétation

- **Étape 1 vide (0 chunk trouvé)** → le contenu n'a jamais été correctement
  extrait/indexé (problème d'extraction PDF, ou fichier pas dans `data/docs/`,
  ou index pas reconstruit après ajout). Il faut vérifier `data/docs/` et relancer
  `python src/build_index.py`.
- **Étape 1 trouve le chunk, mais il n'apparaît pas dans le top 10 de l'étape 2**
  → problème de classement pur. Le contenu du chunk ou de la question ne
  "matchent" pas assez sémantiquement pour ce modèle d'embeddings.
- **Étape 1 et 2 sont bons (chunk présent et bien classé), mais l'assistant se
  trompe quand même** → le problème est dans la génération (prompt), pas dans la
  recherche.